<a href="https://colab.research.google.com/github/rebeccaastaix/Dissertation-Supplementary-Materials/blob/main/SaudiArabiav2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Saudi_Arabia_merged_dataset.xlsx to Saudi_Arabia_merged_dataset.xlsx


In [ ]:
file_name = "Saudi_Arabia_merged_dataset.xlsx"
sheet_name = "Saudi_Model_Data"

df = pd.read_excel(file_name, sheet_name=sheet_name)

print(df.columns.tolist())

['Year', 'Non-Oil Real GDP Growth (%)', 'Oil Price Change', 'Gov Spending Growth', 'FDI net inflows (% of GDP)', 'Non-oil Revenue for General Government (% of Non-oil GDP)', 'Saudi Arabia Total GDP Growth (annual %)', 'Annual Avg Brent Price (USD/bbl)', 'General Government Final Consumption Expenditure (current US$)', 'Lag Non-Oil GDP Growth', 'Non-oil Revenue for General Government']


In [ ]:

df = df[[
    "Year",
    "Non-Oil Real GDP Growth (%)",
    "Oil Price Change",
    "Saudi Arabia Total GDP Growth (annual %)"
]].copy()

In [ ]:
df["LagNonOilGrowth"] = df["Non-Oil Real GDP Growth (%)"].shift(1)
df["LagTotalGrowth"] = df["Saudi Arabia Total GDP Growth (annual %)"].shift(1)

df = df.dropna().copy()

print(df.head())

   Year  Non-Oil Real GDP Growth (%)  Oil Price Change  \
1  2001                     0.134783        -14.323012   
2  2002                     3.102801          1.623926   
3  2003                     3.746447         14.588862   
4  2004                     8.873484         32.851885   
5  2005                     7.561763         43.025704   

   Saudi Arabia Total GDP Growth (annual %)  LagNonOilGrowth  LagTotalGrowth  
1                                  0.344593         0.496231        4.718134  
2                                 -0.688390         0.134783        0.344593  
3                                  8.768734         3.102801       -0.688390  
4                                  8.631005         3.746447        8.768734  
5                                  5.944896         8.873484        8.631005  


In [ ]:
# total GDP


Y_total = df["Saudi Arabia Total GDP Growth (annual %)"]
X_total = df[["Oil Price Change", "LagTotalGrowth"]]
X_total = sm.add_constant(X_total)

model_total = sm.OLS(Y_total, X_total).fit()
print(model_total.summary())

                                       OLS Regression Results                                       
Dep. Variable:     Saudi Arabia Total GDP Growth (annual %)   R-squared:                       0.499
Model:                                                  OLS   Adj. R-squared:                  0.451
Method:                                       Least Squares   F-statistic:                     10.46
Date:                                      Fri, 17 Apr 2026   Prob (F-statistic):           0.000706
Time:                                              21:26:30   Log-Likelihood:                -57.912
No. Observations:                                        24   AIC:                             121.8
Df Residuals:                                            21   BIC:                             125.4
Df Model:                                                 2                                         
Covariance Type:                                  nonrobust                                

In [ ]:
# non oil gdp

Y_nonoil = df["Non-Oil Real GDP Growth (%)"]
X_nonoil = df[["Oil Price Change", "LagNonOilGrowth"]]
X_nonoil = sm.add_constant(X_nonoil)

model_nonoil = sm.OLS(Y_nonoil, X_nonoil).fit()
print(model_nonoil.summary())

                                 OLS Regression Results                                
Dep. Variable:     Non-Oil Real GDP Growth (%)   R-squared:                       0.710
Model:                                     OLS   Adj. R-squared:                  0.682
Method:                          Least Squares   F-statistic:                     25.66
Date:                         Fri, 17 Apr 2026   Prob (F-statistic):           2.30e-06
Time:                                 21:26:46   Log-Likelihood:                -49.837
No. Observations:                           24   AIC:                             105.7
Df Residuals:                               21   BIC:                             109.2
Df Model:                                    2                                         
Covariance Type:                     nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------

In [ ]:
# stress tests

oil_shocks = [-10, -20, -30]

results = []

avg_lag_total = df["LagTotalGrowth"].mean()
avg_lag_nonoil = df["LagNonOilGrowth"].mean()

for shock in oil_shocks:
    pred_total = (
        model_total.params["const"]
        + model_total.params["Oil Price Change"] * shock
        + model_total.params["LagTotalGrowth"] * avg_lag_total
    )

    pred_nonoil = (
        model_nonoil.params["const"]
        + model_nonoil.params["Oil Price Change"] * shock
        + model_nonoil.params["LagNonOilGrowth"] * avg_lag_nonoil
    )

    results.append([shock, pred_total, pred_nonoil])

stress_df = pd.DataFrame(results, columns=[
    "OilShock_pct",
    "Predicted_Total_GDP_Growth",
    "Predicted_NonOil_GDP_Growth"
])

print(stress_df)

   OilShock_pct  Predicted_Total_GDP_Growth  Predicted_NonOil_GDP_Growth
0           -10                    2.068587                     3.869814
1           -20                    1.074536                     2.923653
2           -30                    0.080486                     1.977493


In [ ]:
# fdi test

df_fdi = pd.read_excel(file_name, sheet_name=sheet_name)

df_fdi = df_fdi[[
    "Year",
    "Non-Oil Real GDP Growth (%)",
    "Oil Price Change",
    "FDI net inflows (% of GDP)"
]].copy()

df_fdi["LagNonOilGrowth"] = df_fdi["Non-Oil Real GDP Growth (%)"].shift(1)
df_fdi = df_fdi.dropna().copy()

Y_ext = df_fdi["Non-Oil Real GDP Growth (%)"]
X_ext = df_fdi[["Oil Price Change", "FDI net inflows (% of GDP)", "LagNonOilGrowth"]]
X_ext = sm.add_constant(X_ext)

model_ext = sm.OLS(Y_ext, X_ext).fit()
print(model_ext.summary())

                                 OLS Regression Results                                
Dep. Variable:     Non-Oil Real GDP Growth (%)   R-squared:                       0.716
Model:                                     OLS   Adj. R-squared:                  0.674
Method:                          Least Squares   F-statistic:                     16.82
Date:                         Fri, 17 Apr 2026   Prob (F-statistic):           1.08e-05
Time:                                 21:28:01   Log-Likelihood:                -49.565
No. Observations:                           24   AIC:                             107.1
Df Residuals:                               20   BIC:                             111.8
Df Model:                                    3                                         
Covariance Type:                     nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
-------------------------